# Acquirium client reference

This is the reference guide for the acquirium client: the query interface feature by feature, with the internals (`show_query_graph`, `to_sparql`, text resolution) shown along the way. If you are new, start with `quickstart.ipynb`.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [1]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`_class` accepts a URI or a natural-language string. `alias` names the node for later reference.

In [2]:
q = acq.find_entity(_class="Pump", alias="pump")
_ = q.metadata_head()

Metadata First
   10 Rows    
┏━━━━━━━━━━━━┓
┃ pump       ┃
┡━━━━━━━━━━━━┩
│ wbs:P1     │
│ wbs:P2     │
│ wbs:intake │
└────────────┘

### How strings become URIs
Every string is resolved server-side by an embedding matcher. `resolve_text` shows what a string resolves to — check it when a query returns something unexpected, and pass exact URIs when correctness matters (the top match is not always the intended one):

In [3]:
acq.client.resolve_text("salt", kind="class", top_k=3)

[{'uri': 'urn:nawi-water-ontology#Salt-NaCl',
  'kind': 'class',
  'label': 'Salt-NaCl',
  'score': 0.8903464674949646,
  'matched_surface': 'salt na cl',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#Constituent-Salt',
  'kind': 'class',
  'label': 'Constituent-Salt',
  'score': 0.7883857488632202,
  'matched_surface': 'constituent salt',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#pHAdjuster-Lime',
  'kind': 'class',
  'label': 'Lime',
  'score': 0.757838785648346,
  'matched_surface': 'lime',
  'match_stage': 'semantic',
  'related': []}]

## Follow relationships
`find_related` adds a neighbour reachable within `hops`. `predicates` restricts which edges to follow (`multi_hop_predicates=True` applies them at every hop); `direction="upstream"/"downstream"` walks the S223 piping topology instead. Strings and URIs are both accepted.

In [4]:
q = (
    acq.find_entity(_class="Pump", alias="pump")
       .find_related(_class="Tank", alias="tank", _from="pump", hops=1)
)
q.show_query_graph()
_ = q.metadata_head()

QUERY GRAPH

Nodes:
  0 [pump]  class=http://data.ashrae.org/standard223#Pump
  2 [tank]  class=urn:nawi-water-ontology#Tank

Edges:
  pump --(*, hops=1)--> tank

Data nodes: (none)

Current pointer: tank



           Metadata First 10 Rows            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ tank                         ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:intake │ wbs:ferric-chloride-addition │
└────────────┴──────────────────────────────┘

## Attach data nodes
`find_data` adds the observable/actuatable properties of the current node. `find_all_data` does it for every entity in the graph.


In [5]:
q = acq.find_entity(_class="Pump", alias="pump").find_data()
_ = q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-toc-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-mechanical-power         │
│ wbs:P2     │ wbs:P2-mechanical-power         │
│ wbs:P2     │ wbs:P2-efficiency               │
└────────────┴─────────────────────────────────┘

In [6]:
q_all = acq.find_all_data()
_ =q_all.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:storage-tank-3-out-flow-rate                  │
│ wbs:RO-out-flow-mass-water                        │
│ wbs:P1-out-pressure                               │
│ wbs:conn-cartridge-filtration-to-S1-pressure      │
│ wbs:intake-in-tds-concentration                   │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:RO-out-retentate-flow-mass-water              │
│ wbs:intake-in-flow-rate                           │
│ wbs:PXR-brine-out-pressure                        │
│ wbs:intake-in-tss-concentration                   │
└───────────────────────────────────────────────────┘

## Filter data nodes
Filters apply to the bound data nodes. Strings are resolved via the text matcher.

In [7]:
q = (
    acq.find_all_data()
       .filter_by_quantity_kind("Pressure")
)
_ =q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:RO-in-pressure                           │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-pressure                          │
│ wbs:P1-out-pressure                          │
└──────────────────────────────────────────────┘

In [8]:
q = (
    acq.find_all_data()
       .filter_by_unit("KG/s")
)
_ =q.metadata_head()

                Metadata First 10 Rows                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-retentate-flow-mass-tds                  │
│ wbs:RO-in-flow-mass-tds                             │
│ wbs:RO-out-flow-mass-water                          │
│ wbs:RO-out-retentate-flow-mass-water                │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds   │
│ wbs:PXR-brine-out-flow-mass-tds                     │
└─────────────────────────────────────────────────────┘

In [9]:
q = (
    acq.find_all_data()
        .filter_by_substance("constituent Salt")
        .filter_by_unit("KG/s")
)
_ =q.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:PXR-brine-out-flow-mass-tds                   │
└───────────────────────────────────────────────────┘

The built-in filters map to fixed predicates (`qudt:hasUnit`, `s223:ofSubstance`, `qudt:hasQuantityKind`, `s223:hasMedium`). For any other predicate use `filter_data_nodes` directly (values are exact URIs, no text resolution):

In [10]:
S223 = "http://data.ashrae.org/standard223#"
q_brine = acq.find_all_data().filter_data_nodes(
    predicate=S223 + "ofMedium", value=["urn:nawi-water-ontology#Water-Brine"])
_ = q_brine.metadata_head()

        Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-retentate-flow-mass-tds  │
│ wbs:PXR-brine-out-tds-concentration │
│ wbs:PXR-brine-out-flow-mass-tds     │
└─────────────────────────────────────┘

## Inspect the query
Every query compiles to SPARQL against the server's graph — nothing is hidden. `show_query_graph` prints the node/edge structure, `to_sparql` returns the compiled query (check it when a result surprises you), `metadata` returns the full result as a polars DataFrame (`include_internals=True` keeps the ref/unit columns).

In [11]:
q.show_query_graph()

QUERY GRAPH

Nodes:
  0 [0] [DATA]  class=*

Edges:

Data nodes:
  0 [0]  filters={http://data.ashrae.org/standard223#ofSubstance=['urn:nawi-water-ontology#Constituent-Salt'], http://qudt.org/schema/qudt/hasUnit=['http://qudt.org/vocab/unit/KiloGM-PER-SEC']}}

Current pointer: 0



In [12]:
print(q.to_sparql())

SELECT DISTINCT ?v0 ?ext0 ?unit0 ?extunit0
WHERE {
  ?v0 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext0 .
  OPTIONAL { ?v0 <http://qudt.org/schema/qudt/hasUnit> ?unit0 . }
  OPTIONAL { ?ext0 <http://qudt.org/schema/qudt/hasUnit> ?extunit0 . }
  { { ?v0 <http://data.ashrae.org/standard223#ofSubstance> <urn:nawi-water-ontology#Constituent-Salt> . } }
  { { ?v0 <http://qudt.org/schema/qudt/hasUnit> <http://qudt.org/vocab/unit/KiloGM-PER-SEC> . } }
}


In [13]:
df_meta = q.metadata()
df_meta

0
str
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-in-flow-mass-tds"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"


## Pull timeseries
`dataframe` returns a polars frame. `shape="wide"` puts each data node in its own column, `"narrow"` is long-form. `latest_data` is a shortcut for the most recent point.

In [14]:
end = datetime.now(tz=timezone.utc)
start = end - timedelta(hours=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

data_alias,point_uri,ref_uri,time,value_numeric,value_text
str,str,str,"datetime[μs, UTC]",f64,str


In [15]:
q.latest_data(limit=2)

time,wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,wbs:RO-out-retentate-flow-mass-tds,wbs:PXR-brine-out-flow-mass-tds,wbs:RO-in-flow-mass-tds,wbs:RO-out-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2025-01-30 23:40:00 UTC,9.675431,9.645634,9.645634,9.675431,0.029797
2025-01-30 23:50:00 UTC,9.901141,9.870953,9.870953,9.901141,0.030187


## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [16]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe()

data_alias,point_uri,ref_uri,time,value_numeric,value_text
str,str,str,"datetime[μs, UTC]",f64,str


## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [17]:
data.units()

{'0': 'http://qudt.org/vocab/unit/KiloGM-PER-SEC'}

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [18]:
data = data.convert_to("kg/min")
data.dataframe().head()

data_alias,point_uri,ref_uri,time,value_numeric,value_text
str,str,str,"datetime[μs, UTC]",f64,str


### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [19]:
def list_systems():
    q = acq.find_entity(_class = "System", alias = "Systems")
    return q.metadata()
list_systems()

Systems
str
"""wbs:pretreatment-system"""
"""wbs:desalination-system"""
"""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant"""


The systems are hierarchically organized as:

In [20]:
def list_systems_hier():
    q = acq.find_entity(_class = "System", alias = "Systems")
    q = q.find_related(_class = "System", predicates = ['hasMember'], alias = "Subsystem")
    q.metadata_head()
list_systems_hier()

               Metadata First 10 Rows               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Systems               ┃ Subsystem                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:seawater-ro-plant │ wbs:pretreatment-system  │
│ wbs:seawater-ro-plant │ wbs:desalination-system  │
│ wbs:seawater-ro-plant │ wbs:posttreatment-system │
└───────────────────────┴──────────────────────────┘

We can see how many equipment we have in each system:

In [21]:
def list_equipment_by_system(hops = 1):
        q = acq.find_entity(_class = "System", alias = "Systems")
        q = q.find_related(_class = "Equipment", predicates = ['hasMember'], alias = "Equipment", hops = hops, multi_hop_predicates = True)
        q_df = q.metadata()
        return q_df.group_by("Systems").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count", descending=True)

list_equipment_by_system()

Systems,equipment_count
str,u32
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


These are the number of equipment directly a member of these systems.

If we increase the hops, you'll see total number equipments in each system

In [22]:
list_equipment_by_system(3)

Systems,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


Let's find the pumps in a specific system:


In [23]:
def list_equipment_in_system(system, equipment):
    q = acq.find_entity(uri=system, alias = "system").find_related(_class=equipment, alias = "equipment", hops=1)
    q.metadata_head()

list_equipment_in_system('wbs:pretreatment-system', 'pump')

         Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ system                  ┃ equipment  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ wbs:pretreatment-system │ wbs:intake │
└─────────────────────────┴────────────┘

Let's find all the pumps and their data

In [24]:
def all_pumps_and_their_data():
    q = acq.find_entity(_class="pump", alias="pump").find_all_data()
    q.metadata_head()
    return q.data(limit=10).dataframe()

all_pumps_and_their_data().head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-toc-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-mechanical-power         │
│ wbs:P2     │ wbs:P2-mechanical-power         │
│ wbs:P2     │ wbs:P2-efficiency               │
└────────────┴─────────────────────────────────┘

time,pump_data__wbs:intake-in-flow-rate,pump_data__wbs:intake-in-tds-concentration,pump_data__wbs:intake-in-toc-concentration,pump_data__wbs:P1-efficiency,pump_data__wbs:P1-mechanical-power,pump_data__wbs:P2-efficiency,pump_data__wbs:intake-in-tss-concentration,pump_data__wbs:P1-out-pressure,pump_data__wbs:P2-mechanical-power
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-01-01 00:00:00 UTC,0.300595,33.537931,0.003172,0.8,1.1993e6,0.8,0.035273,7e6,113273.920254
2025-01-01 00:10:00 UTC,0.287638,33.438568,0.002494,0.8,1.1789e6,0.8,0.034358,7e6,102902.022309
2025-01-01 00:20:00 UTC,0.292075,33.410724,0.002606,0.8,1.1898e6,0.8,0.023289,7e6,106420.418542
2025-01-01 00:30:00 UTC,0.290283,33.599669,0.002706,0.8,1.1816e6,0.8,0.02101,7e6,105524.665282
2025-01-01 00:40:00 UTC,0.299641,33.491902,0.002696,0.8,1.1998e6,0.8,0.02581,7e6,112484.749591


Let's find all the data generating entites within a system:

In [25]:
def find_all_sensors(system):
    q = (acq.find_entity(uri=system, alias="backwash")
         .find_related(_class="equipment", alias="equipment",predicates=['hasMember'], hops=1)
         .find_data(alias = "sensors"))
    q_df = q.metadata(include_internals=True)
    q_df = q_df.drop([pl.col('backwash'),pl.col('sensors_ref'),pl.col('extunit4')])
    return q_df

system = 'wbs:pretreatment-system'
find_all_sensors(system)

equipment,sensors,unit4
str,str,str
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:intake""","""wbs:intake-in-toc-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
